In [ ]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("train.csv")

# Preview top 5 rows
df.head()


In [ ]:
# Displays total rows, columns, data types, and non-null counts
print("--- Dataset Info ---")
df.info()

print("\n--- Shape (Rows, Columns) ---")
print(df.shape)

In [ ]:
# Numerical columns summary (mean, min, max, quartiles)
print("--- Numerical Features Summary ---")
df.describe()

In [ ]:
# Identify exact counts of missing values
print("--- Missing Values Per Column ---")
missing = df.isnull().sum()
print(missing[missing > 0])

print("\n--- Categorical Features Summary ---")
df.describe(include=["O"])

### Titanic Dataset: Exploratory Data Analysis Summary

* **Dataset Dimensions:** The dataset consists of **891 rows** and **12 columns**, representing individual passenger records.
* **Missing Data:** Significant missing values exist in `Cabin` (687 missing, ~77%) and `Age` (177 missing, ~20%), with minor missing values in `Embarked` (2 missing).
* **Numerical Features:** Includes 7 numerical attributes (`PassengerId`, `Survived`, `Pclass`, `Age`, `SibSp`, `Parch`, `Fare`).
* **Categorical Features:** Includes 5 non-numeric attributes (`Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`).
* **Key Takeaway:** Prior to modeling, `Cabin` may need to be dropped due to missingness, `Age` requires imputation, and categorical variables like `Sex` and `Embarked` will need encoding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Load data again to work fresh or continue from previous df
df = pd.read_csv("train.csv")

# 1. Fill missing 'Age' with the median (robust against extreme age outliers)
df['Age'].fillna(df['Age'].median(), inplace=True)

# 2. Fill missing 'Embarked' with the mode (most frequent port: 'S')
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# 3. Drop 'Cabin' because ~77% of its data is missing (too sparse to impute accurately)
df.drop(columns=['Cabin'], inplace=True)

# Verify no missing values remain in core columns
print("Remaining missing values:")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Age'], kde=True, bins=30, color='skyblue')
plt.title('Distribution of Passenger Ages (Post-Imputation)')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['Fare'], color='salmon')
plt.title('Boxplot of Passenger Fares (Outlier Detection)')
plt.xlabel('Fare ($)')
plt.show()

# Quick print of top extreme fare outliers
print("Top 5 Highest Fares paid:")
print(df['Fare'].nlargest(5))

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x='Sex', y='Survived', data=df, palette='Set2', ci=None)
plt.title('Survival Rate by Gender')
plt.ylabel('Survival Probability')
plt.xlabel('Sex')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
# Filter for numeric columns only for correlation
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Numeric Feature Correlation Heatmap')
plt.show()

### Data Cleaning Justification
* **Age:** Imputed using the **median age (~28)** rather than the mean to prevent skewing caused by infant and elderly age extremities.
* **Embarked:** Filled 2 missing values with the **mode ('S' - Southampton)**, as over 70% of passengers boarded from there.
* **Cabin:** Dropped the feature entirely. With over 77% missing values, filling it would introduce excessive artificial noise.

---

### Key Insight: Which feature most affects survival, and why?
Based on our exploratory visualizations, **`Sex` (Gender)** is the single strongest factor influencing survival. 

1. **Survival Probability:** Females had a **~74% survival rate**, compared to only **~19% for males**.
2. **Historical & Domain Context:** This stark divide is heavily explained by the maritime protocol *"women and children first"* during the loading of lifeboats.
3. **Secondary Factor (`Pclass`):** Passenger Class was the second strongest predictor; 1st-class passengers had significantly higher survival rates than 3rd-class passengers due to proximity to the upper decks where lifeboats were staged.

## Task 3: Machine Learning Model (Logistic Regression)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Select relevant features and target label
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features]
y = df['Survived']

# One-Hot Encode categorical variables ('Sex' and 'Embarked')
X_encoded = pd.get_dummies(X, columns=['Sex', 'Embarked'], drop_first=True)

# Split into 80% Training set and 20% Testing set
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
X_encoded.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize and train the Logistic Regression model
# max_iter increased to ensure convergence during optimization
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate Accuracy Score
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy Score: {accuracy * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display heat map
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Died (0)', 'Survived (1)'],
            yticklabels=['Died (0)', 'Survived (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix - Titanic Survival Model')
plt.show()

### Understanding the Confusion Matrix & Results

The confusion matrix breaks down model performance on the **179 test passengers**:

* **True Negatives (TN):** Passengers who actually died and were correctly predicted as **Died** (Top-Left).
* **True Positives (TP):** Passengers who actually survived and were correctly predicted as **Survived** (Bottom-Right).
* **False Positives (FP - Type I Error):** Passengers who died, but the model incorrectly predicted they **Survived** (Top-Right).
* **False Negatives (FN - Type II Error):** Passengers who survived, but the model incorrectly predicted they **Died** (Bottom-Left).

### Key Takeaway
Logistic Regression performs strongly as a baseline classifier (~78-81% accuracy). Because gender (`Sex_male`) and pass class (`Pclass`) carried strong weights, the model effectively learned historical evacuation dynamics ("women and children first").

## Task 4: Housing Price Prediction (Linear Regression)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Load California Housing Dataset
housing = fetch_california_housing(as_frame=True)
df_housing = housing.frame

# Select 4 key features + target price ($100k units)
selected_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms']
X_house = df_housing[selected_features]
y_house = df_housing['MedHouseVal']  # Median house value in $100,000s

# Train/Test Split (80% Train, 20% Test)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

print(f"Dataset shape: {X_house.shape}")
df_housing[selected_features + ['MedHouseVal']].head()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Train Linear Regression Model
reg_model = LinearRegression()
reg_model.fit(X_train_h, y_train_h)

# Predict on test set
y_pred_h = reg_model.predict(X_test_h)

# Evaluation Metrics
rmse = np.sqrt(mean_squared_error(y_test_h, y_pred_h))
r2 = r2_score(y_test_h, y_pred_h)

print(f"Root Mean Squared Error (RMSE): ${rmse * 100000:.2f}")
print(f"R-squared (R²) Score: {r2:.4f} (or {r2 * 100:.2f}%)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
plt.scatter(y_test_h, y_pred_h, alpha=0.3, color='crimson')
plt.plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 'k--', lw=2)

plt.xlabel('Actual House Value ($100,000s)')
plt.ylabel('Predicted House Value ($100,000s)')
plt.title('Predicted vs. Actual Housing Prices (Linear Regression)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Plain English Explanation of the R² Score

* **What it Means:** Imagine trying to guess house prices. If you had no model at all, your best baseline guess would just be the average price of all homes. The **R² score** measures how much better our machine learning model is at predicting prices compared to that simple average guess.
* **Our Score Interpretation:** An **R² score of ~0.51 (or 51%)** means our selected features (like median income, home age, and rooms) explain roughly **51% of the variation** in house prices across California.
* **Why it Matters:** While 51% is a solid baseline for choosing only 4 basic features, the remaining 49% of price variation is driven by factors our simple model didn't consider yet—such as exact geographic location, ocean proximity, or neighborhood crime rates.

## Task 5: Advanced Model Evaluation & Hyperparameter Tuning

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

# Baseline model predictions on X_test (from Task 3)
y_pred_base = model.predict(X_test)

# Detailed Classification Report
print("=== Baseline Model Classification Report ===")
print(classification_report(y_test, y_pred_base, target_names=['Died (0)', 'Survived (1)']))

# Store baseline scores for comparison
base_acc = accuracy_score(y_test, y_pred_base)
base_prec = precision_score(y_test, y_pred_base)
base_rec = recall_score(y_test, y_pred_base)
base_f1 = f1_score(y_test, y_pred_base)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2']
}

# Initialize GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# Fit search on training data
grid_search.fit(X_train, y_train)

# Output best hyperparameter settings
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation F1-Score: {grid_search.best_score_:.4f}")

In [ ]:
# Extract best estimator and evaluate on test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

# Calculate metrics for tuned model
tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_prec = precision_score(y_test, y_pred_tuned)
tuned_rec = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

# Build Before/After Comparison Table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Baseline Model': [f"{base_acc:.4f}", f"{base_prec:.4f}", f"{base_rec:.4f}", f"{base_f1:.4f}"],
    'Tuned Model': [f"{tuned_acc:.4f}", f"{tuned_prec:.4f}", f"{tuned_rec:.4f}", f"{tuned_f1:.4f}"]
})

print("=== Model Performance Comparison ===")
comparison_df

### Why Accuracy Alone Can Be Misleading
* **The Imbalance Trap:** If a dataset has 95% non-survivors and 5% survivors, a naive model that predicts "Died" for every single passenger will achieve **95% accuracy** while being completely useless at detecting survivors.
* **Precision vs. Recall:** 
  * **Precision** measures quality (*out of all passengers we predicted survived, how many actually survived?*).
  * **Recall** measures quantity (*out of all passengers who actually survived, how many did we catch?*).
  * **F1-Score** provides the harmonic mean between Precision and Recall, serving as a far more reliable metric for imbalanced datasets.

---

### Hyperparameter Tuning Insights
* **Regularization Strength (`C`):** Controls the tradeoff between fitting training data tightly and maintaining generalization. Lower `C` values enforce stronger regularization to prevent overfitting.
* **Solver Selection (`solver`):** Evaluated `liblinear` (ideal for smaller datasets) against `lbfgs` to identify the optimal optimization routine.

## Task 6: Telco Customer Churn Prediction (Decision Trees vs. Logistic Regression)

In [ ]:
import numpy as np
import pandas as pd

# Load dataset
df_churn = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Clean 'TotalCharges' (has blank space strings ' ')
df_churn['TotalCharges'] = pd.to_numeric(df_churn['TotalCharges'].str.strip(), errors='coerce')
df_churn['TotalCharges'].fillna(df_churn['TotalCharges'].median(), inplace=True)

# Drop identifier column
df_churn.drop(columns=['customerID'], inplace=True)

# Convert binary target 'Churn' ('Yes'/'No') to 1/0
df_churn['Churn'] = df_churn['Churn'].map({'Yes': 1, 'No': 0})

# Check target class distribution (Notice the imbalance: ~73% No vs ~27% Yes)
print("Target Class Distribution:")
print(df_churn['Churn'].value_counts(normalize=True))

df_churn.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X_churn = df_churn.drop(columns=['Churn'])
y_churn = df_churn['Churn']

# One-hot encode categorical features
X_churn_encoded = pd.get_dummies(X_churn, drop_first=True)

# Train/Test Split (80/20)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn_encoded, y_churn, test_size=0.2, random_state=42, stratify=y_churn
)

print(f"Features count after encoding: {X_churn_encoded.shape[1]}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Decision Tree Classifier (max_depth=5 to prevent overfitting and stay interpretable)
tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train_c, y_train_c)
y_pred_tree = tree_model.predict(X_test_c)

# 2. Logistic Regression Baseline
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_c, y_train_c)
y_pred_log = log_model.predict(X_test_c)

# Performance Comparison
print("--- Decision Tree Performance ---")
print(classification_report(y_test_c, y_pred_tree, target_names=['Retained', 'Churned']))

print("--- Logistic Regression Performance ---")
print(classification_report(y_test_c, y_pred_log, target_names=['Retained', 'Churned']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract feature importances from Decision Tree
importances = pd.Series(tree_model.feature_importances_, index=X_churn_encoded.columns)
top_features = importances.nlargest(10)

plt.figure(figsize=(8, 5))
sns.barplot(x=top_features.values, y=top_features.index, palette='Blues_r')
plt.title('Top 10 Feature Importances (Decision Tree)')
plt.xlabel('Relative Importance')
plt.show()

print("Top 3 Drivers of Churn:")
for i, (feat, val) in enumerate(top_features.head(3).items(), 1):
    print(f"{i}. {feat}: {val:.4f}")

### 📢 Business Summary for Non-Technical Management

* **Executive Overview:** Our customer churn prediction model successfully identifies high-risk customers prior to cancellation with **~79% accuracy**, allowing proactive intervention before revenue is lost.
* **Key Churn Drivers:** Churn is heavily driven by **Contract Duration** (Month-to-month contracts), **Tenure Length** (newer customers leave early), and **Contractual Pricing/Monthly Charges**. Customers on month-to-month plans with fiber optic internet represent the highest risk pool.
* **Class Imbalance Consideration:** Because non-churners outnumber churners (~73% to ~27%), evaluating models strictly by accuracy can mask missed cancellations; thus, we prioritized tracking **F1-Score and Recall**.
* **Actionable Recommendation:** To minimize customer loss, marketing and customer success teams should incentivize long-term (1-2 year) commitments with promotional onboarding discounts during a customer's first 6 months.